In [0]:
# Databricks notebook source
# /// script
# [tool.databricks.environment]
# environment_version = "5"
# ///
# DBTITLE 1,Overview
# MAGIC %md
# MAGIC # Gold Layer — UC Metric Views
# MAGIC
# MAGIC Standardized KPIs for the aircraft predictive maintenance semantic layer.
# MAGIC - **Fleet Operations** — flight counts, anomaly rates, fuel efficiency
# MAGIC - **Aircraft Health** — risk scores, maintenance urgency, fleet status
# MAGIC - **Anomaly Analysis** — event frequency, severity, duration by type
# MAGIC
# MAGIC All views created in `genie_zeroops_mfg_catalog.default` on top of the gold medallion tables.

# COMMAND ----------

# DBTITLE 1,Fleet Operations Metrics
# MAGIC %sql
# MAGIC CREATE OR REPLACE VIEW genie_zeroops_mfg_catalog.default.fleet_operations_metrics
# MAGIC WITH METRICS
# MAGIC LANGUAGE YAML
# MAGIC AS $$
# MAGIC   version: 1.1
# MAGIC   source: genie_zeroops_mfg_catalog.default.gold_flight_summary
# MAGIC   comment: Fleet-wide operational KPIs aggregated from the gold flight summary layer.
# MAGIC   dimensions:
# MAGIC     - name: airline
# MAGIC       expr: airline
# MAGIC       comment: Operating airline
# MAGIC       synonyms:
# MAGIC         - carrier
# MAGIC         - operator
# MAGIC     - name: aircraft_type
# MAGIC       expr: aircraft_type
# MAGIC       comment: Aircraft model designation
# MAGIC       synonyms:
# MAGIC         - aircraft model
# MAGIC         - plane type
# MAGIC     - name: origin
# MAGIC       expr: origin
# MAGIC       comment: Departure airport IATA code
# MAGIC     - name: destination
# MAGIC       expr: destination
# MAGIC       comment: Arrival airport IATA code
# MAGIC     - name: anomaly_type
# MAGIC       expr: anomaly_type
# MAGIC       comment: Type of anomaly detected (null for normal flights)
# MAGIC     - name: anomaly_severity
# MAGIC       expr: anomaly_severity
# MAGIC       comment: Severity of anomaly (LOW/MEDIUM/HIGH/CRITICAL)
# MAGIC     - name: has_anomaly
# MAGIC       expr: CAST(has_anomaly AS STRING)
# MAGIC       comment: Whether the flight had any anomaly
# MAGIC     - name: flight_month
# MAGIC       expr: DATE_TRUNC('MONTH', departure_time)
# MAGIC       display_name: Flight Month
# MAGIC       comment: Month of departure
# MAGIC       format:
# MAGIC         type: date
# MAGIC         date_format: locale_short_month
# MAGIC   measures:
# MAGIC     - name: Total Flights
# MAGIC       expr: COUNT(1)
# MAGIC       display_name: Total Flights
# MAGIC       comment: Total number of flights
# MAGIC       format:
# MAGIC         type: number
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 0
# MAGIC     - name: Anomaly Count
# MAGIC       expr: COUNT(1) FILTER (WHERE has_anomaly = true)
# MAGIC       display_name: Anomaly Count
# MAGIC       comment: Number of flights with detected anomalies
# MAGIC     - name: Anomaly Rate
# MAGIC       expr: COUNT(1) FILTER (WHERE has_anomaly = true) / NULLIF(CAST(COUNT(1) AS DOUBLE), 0)
# MAGIC       display_name: Anomaly Rate
# MAGIC       comment: Fraction of flights with anomalies
# MAGIC       format:
# MAGIC         type: percentage
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 1
# MAGIC     - name: Avg Flight Duration Hours
# MAGIC       expr: AVG(flight_duration_hours)
# MAGIC       display_name: Avg Flight Duration (hrs)
# MAGIC       comment: Average flight duration in hours
# MAGIC       format:
# MAGIC         type: number
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 2
# MAGIC     - name: Total Fuel Burned
# MAGIC       expr: SUM(total_fuel_burned)
# MAGIC       display_name: Total Fuel Burned
# MAGIC       comment: Aggregate fuel consumption across all flights
# MAGIC       format:
# MAGIC         type: number
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 0
# MAGIC         abbreviation: compact
# MAGIC     - name: Avg Max Vibration
# MAGIC       expr: AVG(max_vibration)
# MAGIC       display_name: Avg Peak Vibration
# MAGIC       comment: Average of per-flight maximum vibration readings
# MAGIC       format:
# MAGIC         type: number
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 3
# MAGIC     - name: Avg Min Hydraulic Pressure
# MAGIC       expr: AVG(min_hydraulic_pressure)
# MAGIC       display_name: Avg Min Hydraulic Pressure (psi)
# MAGIC       comment: Average of per-flight minimum hydraulic system pressure
# MAGIC       format:
# MAGIC         type: number
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 0
# MAGIC     - name: Avg Max EGT
# MAGIC       expr: AVG(GREATEST(max_egt_eng1, max_egt_eng2))
# MAGIC       display_name: Avg Peak EGT (°C)
# MAGIC       comment: Average of the higher per-flight EGT across both engines
# MAGIC       format:
# MAGIC         type: number
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 1
# MAGIC     - name: Avg Cruise Speed
# MAGIC       expr: AVG(avg_cruise_speed)
# MAGIC       display_name: Avg Cruise Speed (kts)
# MAGIC       comment: Average cruise-phase airspeed
# MAGIC       format:
# MAGIC         type: number
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 1
# MAGIC $$

# COMMAND ----------

# DBTITLE 1,Aircraft Health Metrics
# MAGIC %sql
# MAGIC CREATE OR REPLACE VIEW genie_zeroops_mfg_catalog.default.aircraft_health_metrics
# MAGIC WITH METRICS
# MAGIC LANGUAGE YAML
# MAGIC AS $$
# MAGIC   version: 1.1
# MAGIC   source: genie_zeroops_mfg_catalog.default.ml_maintenance_predictions
# MAGIC   comment: Per-aircraft health and maintenance priority KPIs combining ML predictions with operational health scores.
# MAGIC   joins:
# MAGIC     - name: health
# MAGIC       source: genie_zeroops_mfg_catalog.default.gold_aircraft_health
# MAGIC       on: source.tail_number = health.tail_number
# MAGIC       cardinality: many_to_one
# MAGIC       rely:
# MAGIC         at_most_one_match: true
# MAGIC   dimensions:
# MAGIC     - name: airline
# MAGIC       expr: source.airline
# MAGIC       comment: Operating airline
# MAGIC     - name: aircraft_type
# MAGIC       expr: source.aircraft_type
# MAGIC       comment: Aircraft model designation
# MAGIC     - name: risk_category
# MAGIC       expr: source.risk_category
# MAGIC       display_name: Risk Category
# MAGIC       comment: ML-derived maintenance risk tier (LOW/MEDIUM/HIGH/CRITICAL)
# MAGIC       synonyms:
# MAGIC         - risk level
# MAGIC         - priority tier
# MAGIC     - name: aircraft_age_years
# MAGIC       expr: health.aircraft_age_years
# MAGIC       display_name: Aircraft Age (years)
# MAGIC       comment: Years since manufacture
# MAGIC   measures:
# MAGIC     - name: Fleet Size
# MAGIC       expr: COUNT(DISTINCT source.tail_number)
# MAGIC       display_name: Fleet Size
# MAGIC       comment: Number of distinct aircraft
# MAGIC       format:
# MAGIC         type: number
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 0
# MAGIC     - name: Critical Aircraft
# MAGIC       expr: COUNT(1) FILTER (WHERE source.risk_category = 'CRITICAL')
# MAGIC       display_name: Critical Aircraft
# MAGIC       comment: Aircraft in CRITICAL maintenance risk tier
# MAGIC     - name: High Risk Aircraft
# MAGIC       expr: COUNT(1) FILTER (WHERE source.risk_category IN ('CRITICAL', 'HIGH'))
# MAGIC       display_name: High + Critical Aircraft
# MAGIC       comment: Aircraft needing priority maintenance attention
# MAGIC     - name: Avg Maintenance Priority
# MAGIC       expr: AVG(source.maintenance_priority_score)
# MAGIC       display_name: Avg Maintenance Priority Score
# MAGIC       comment: Average composite priority score (0–100)
# MAGIC       format:
# MAGIC         type: number
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 1
# MAGIC     - name: Avg Health Risk Score
# MAGIC       expr: AVG(source.health_risk_score)
# MAGIC       display_name: Avg Health Risk Score
# MAGIC       comment: Average operational health risk from the gold layer
# MAGIC       format:
# MAGIC         type: number
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 1
# MAGIC     - name: Avg ML Anomaly Probability
# MAGIC       expr: AVG(source.ml_anomaly_probability)
# MAGIC       display_name: Avg ML Anomaly Probability
# MAGIC       comment: Average binary classifier anomaly probability
# MAGIC       format:
# MAGIC         type: percentage
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 1
# MAGIC     - name: Avg Anomaly Rate
# MAGIC       expr: AVG(source.anomaly_rate)
# MAGIC       display_name: Avg Anomaly Rate
# MAGIC       comment: Average historical anomaly rate per aircraft
# MAGIC       format:
# MAGIC         type: percentage
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 1
# MAGIC     - name: Avg Days Until Maintenance
# MAGIC       expr: AVG(source.days_until_next_maintenance)
# MAGIC       display_name: Avg Days Until Maintenance
# MAGIC       comment: Average days remaining before next scheduled maintenance
# MAGIC       format:
# MAGIC         type: number
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 0
# MAGIC     - name: Overdue Maintenance Count
# MAGIC       expr: COUNT(1) FILTER (WHERE source.days_until_next_maintenance <= 0)
# MAGIC       display_name: Overdue Maintenance
# MAGIC       comment: Aircraft past their scheduled maintenance date
# MAGIC $$

# COMMAND ----------

# DBTITLE 1,Anomaly Analysis Metrics
# MAGIC %sql
# MAGIC CREATE OR REPLACE VIEW genie_zeroops_mfg_catalog.default.anomaly_analysis_metrics
# MAGIC WITH METRICS
# MAGIC LANGUAGE YAML
# MAGIC AS $$
# MAGIC   version: 1.1
# MAGIC   source: genie_zeroops_mfg_catalog.default.gold_anomaly_events
# MAGIC   comment: Anomaly event analysis KPIs for root-cause investigation and trend tracking.
# MAGIC   dimensions:
# MAGIC     - name: anomaly_type
# MAGIC       expr: anomaly_type
# MAGIC       display_name: Anomaly Type
# MAGIC       comment: Category of anomaly detected
# MAGIC       synonyms:
# MAGIC         - fault type
# MAGIC         - failure mode
# MAGIC     - name: anomaly_severity
# MAGIC       expr: anomaly_severity
# MAGIC       display_name: Severity
# MAGIC       comment: Anomaly severity level (LOW/MEDIUM/HIGH/CRITICAL)
# MAGIC     - name: airline
# MAGIC       expr: airline
# MAGIC       comment: Operating airline
# MAGIC     - name: aircraft_type
# MAGIC       expr: aircraft_type
# MAGIC       comment: Aircraft model
# MAGIC     - name: flight_phase_at_peak
# MAGIC       expr: flight_phase_at_peak
# MAGIC       display_name: Peak Flight Phase
# MAGIC       comment: Flight phase where the anomaly parameter deviated most
# MAGIC   measures:
# MAGIC     - name: Total Anomaly Events
# MAGIC       expr: COUNT(1)
# MAGIC       display_name: Total Anomaly Events
# MAGIC       comment: Number of distinct anomaly occurrences
# MAGIC       format:
# MAGIC         type: number
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 0
# MAGIC     - name: Distinct Aircraft Affected
# MAGIC       expr: COUNT(DISTINCT tail_number)
# MAGIC       display_name: Affected Aircraft
# MAGIC       comment: Number of unique aircraft that experienced anomalies
# MAGIC     - name: Avg Event Duration Seconds
# MAGIC       expr: AVG(event_duration_seconds)
# MAGIC       display_name: Avg Event Duration (sec)
# MAGIC       comment: Average duration of anomaly events in seconds
# MAGIC       format:
# MAGIC         type: number
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 0
# MAGIC     - name: Avg Peak EGT
# MAGIC       expr: AVG(peak_egt)
# MAGIC       display_name: Avg Peak EGT (°C)
# MAGIC       comment: Average exhaust gas temperature at anomaly peak
# MAGIC       format:
# MAGIC         type: number
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 1
# MAGIC     - name: Avg Peak Vibration
# MAGIC       expr: AVG(peak_vibration)
# MAGIC       display_name: Avg Peak Vibration
# MAGIC       comment: Average vibration reading at anomaly peak
# MAGIC       format:
# MAGIC         type: number
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 3
# MAGIC     - name: Avg Min Oil Pressure
# MAGIC       expr: AVG(min_oil_pressure_during_event)
# MAGIC       display_name: Avg Min Oil Pressure (psi)
# MAGIC       comment: Average minimum oil pressure observed during anomaly events
# MAGIC       format:
# MAGIC         type: number
# MAGIC         decimal_places:
# MAGIC           type: exact
# MAGIC           places: 1
# MAGIC     - name: Critical Severity Count
# MAGIC       expr: COUNT(1) FILTER (WHERE anomaly_severity = 'CRITICAL')
# MAGIC       display_name: Critical Events
# MAGIC       comment: Anomaly events with CRITICAL severity
# MAGIC     - name: High Severity Count
# MAGIC       expr: COUNT(1) FILTER (WHERE anomaly_severity IN ('CRITICAL', 'HIGH'))
# MAGIC       display_name: High + Critical Events
# MAGIC       comment: Anomaly events with HIGH or CRITICAL severity
# MAGIC $$

# COMMAND ----------

# DBTITLE 1,Validate Metric Views
# MAGIC %sql
# MAGIC -- Validate all three metric views
# MAGIC
# MAGIC SELECT 'fleet_operations_metrics' AS metric_view,
# MAGIC        airline,
# MAGIC        MEASURE(`Total Flights`) AS total_flights,
# MAGIC        MEASURE(`Anomaly Rate`) AS anomaly_rate,
# MAGIC        MEASURE(`Avg Flight Duration Hours`) AS avg_duration
# MAGIC FROM genie_zeroops_mfg_catalog.default.fleet_operations_metrics
# MAGIC GROUP BY airline
# MAGIC ORDER BY total_flights DESC

# COMMAND ----------

# DBTITLE 1,Validate Aircraft Health Metrics
# MAGIC %sql
# MAGIC SELECT risk_category,
# MAGIC        MEASURE(`Fleet Size`) AS fleet_size,
# MAGIC        MEASURE(`Avg Maintenance Priority`) AS avg_priority,
# MAGIC        MEASURE(`Avg ML Anomaly Probability`) AS avg_ml_prob,
# MAGIC        MEASURE(`Avg Days Until Maintenance`) AS avg_days_to_maint
# MAGIC FROM genie_zeroops_mfg_catalog.default.aircraft_health_metrics
# MAGIC GROUP BY risk_category
# MAGIC ORDER BY avg_priority DESC

# COMMAND ----------

# DBTITLE 1,Validate Anomaly Analysis Metrics
# MAGIC %sql
# MAGIC SELECT anomaly_type,
# MAGIC        anomaly_severity,
# MAGIC        MEASURE(`Total Anomaly Events`) AS events,
# MAGIC        MEASURE(`Distinct Aircraft Affected`) AS aircraft_affected,
# MAGIC        MEASURE(`Avg Event Duration Seconds`) AS avg_duration_sec,
# MAGIC        MEASURE(`Avg Peak Vibration`) AS avg_peak_vib
# MAGIC FROM genie_zeroops_mfg_catalog.default.anomaly_analysis_metrics
# MAGIC GROUP BY anomaly_type, anomaly_severity
# MAGIC ORDER BY events DESC